# LSTM A Historical Demand

## 1. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 1
# LOAD DATASET 4 (2019)
# HISTORICAL CHARGING DEMAND ONLY
# ============================================================

from google.colab import drive
import os
import numpy as np
import pandas as pd
import tensorflow as tf

# Mount Google Drive
drive.mount("/content/drive")

# Project directory
ROOT = "/content/drive/MyDrive/EV_Load_Forecasting_FYP"

# Dataset 4 - processed 2019 hourly dataset
dataset_path_f = os.path.join(
    ROOT,
    "Processed Data",
    "Hourly",
    "experiment_c_final_2019.csv"
)

# Experiment F directory
experiment_dir_f = os.path.join(
    ROOT,
    "Experiments",
    "LSTM_Experiment_F_Historical_Only_2019"
)

os.makedirs(experiment_dir_f, exist_ok=True)

# Load dataset
df_f = pd.read_csv(
    dataset_path_f,
    parse_dates=["date_hour"]
)

df_f = (
    df_f.sort_values("date_hour")
    .reset_index(drop=True)
)

# Keep only timestamp and historical demand
df_f = df_f[
    ["date_hour", "hourly_energy_kwh"]
].copy()

print("=" * 70)
print("EXPERIMENT F - DATASET LOADING")
print("=" * 70)

print("\nDataset shape:", df_f.shape)

print("\nDate range:")
print("Start:", df_f["date_hour"].min())
print("End  :", df_f["date_hour"].max())

print("\nColumns:")
print(df_f.columns.tolist())

print("\nMissing values:")
print(df_f.isna().sum())

print("\nFirst 5 rows:")
print(df_f.head())

print("\nExperiment directory:")
print(experiment_dir_f)


## 2. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 2
# CHRONOLOGICAL DATA SPLITTING (70/15/15)
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("EXPERIMENT F - CHRONOLOGICAL DATA SPLITTING")
print("=" * 70)

# 1. Check timestamp quality
duplicate_count_f = df_f["date_hour"].duplicated().sum()

expected_dates_f = pd.date_range(
    start=df_f["date_hour"].min(),
    end=df_f["date_hour"].max(),
    freq="h"
)

missing_timestamps_f = expected_dates_f.difference(
    pd.DatetimeIndex(df_f["date_hour"])
)

print("\nDuplicate timestamps:", duplicate_count_f)
print("Missing hourly timestamps:", len(missing_timestamps_f))

assert duplicate_count_f == 0
assert len(missing_timestamps_f) == 0

# 2. Define chronological split boundaries
n_total_f = len(df_f)

train_end_f = int(n_total_f * 0.70)
val_end_f = int(n_total_f * 0.85)

# 3. Split without shuffling
train_df_f = df_f.iloc[:train_end_f].copy()
val_df_f = df_f.iloc[train_end_f:val_end_f].copy()
test_df_f = df_f.iloc[val_end_f:].copy()

# 4. Display split information
for name, split in [
    ("Training", train_df_f),
    ("Validation", val_df_f),
    ("Testing", test_df_f)
]:
    print(f"\n{name}")
    print("Rows:", len(split))
    print("Start:", split["date_hour"].min())
    print("End:", split["date_hour"].max())
    print(
        "Missing demand values:",
        split["hourly_energy_kwh"].isna().sum()
    )

# 5. Verify split sizes
assert len(train_df_f) == 5611
assert len(val_df_f) == 1202
assert len(test_df_f) == 1203

assert (
    len(train_df_f) +
    len(val_df_f) +
    len(test_df_f)
) == len(df_f)

print("\nChronological splitting completed successfully.")


## 3. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 3
# MISSING VALUE HANDLING
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("EXPERIMENT F - MISSING VALUE HANDLING")
print("=" * 70)

# 1. Preserve original chronological dataset
df_processed_f = df_f.copy()

# 2. Preserve original observed target values
df_processed_f["target_original"] = (
    df_processed_f["hourly_energy_kwh"]
)

# 3. Forward-fill historical demand for LSTM inputs
df_processed_f["demand_input"] = (
    df_processed_f["hourly_energy_kwh"].ffill()
)

# 4. Check missing values
print("\nMissing values after processing:")
print(
    df_processed_f[
        ["target_original", "demand_input"]
    ].isna().sum()
)

# 5. Verify
assert (
    df_processed_f["target_original"].isna().sum()
    == 206
)

assert (
    df_processed_f["demand_input"].isna().sum()
    == 0
)

# 6. Display sample of filled observations
missing_examples_f = df_processed_f.loc[
    df_processed_f["target_original"].isna(),
    ["date_hour", "target_original", "demand_input"]
]

print("\nFirst 10 filled historical observations:")
print(missing_examples_f.head(10).to_string(index=False))

print("\nMissing value handling completed successfully.")


## 4. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 4
# FEATURE SCALING (TRAINING DATA ONLY)
# ============================================================

import os
import joblib
import numpy as np

from sklearn.preprocessing import MinMaxScaler

print("=" * 70)
print("EXPERIMENT F - FEATURE SCALING")
print("=" * 70)

# 1. Define input feature
feature_cols_f = ["demand_input"]

# 2. Initialize scalers
scaler_X_f = MinMaxScaler()
scaler_y_f = MinMaxScaler()

# 3. Fit using training data only
scaler_X_f.fit(
    df_processed_f.iloc[:train_end_f][feature_cols_f]
)

train_targets_f = (
    df_processed_f.iloc[:train_end_f][["target_original"]]
    .dropna()
)

scaler_y_f.fit(train_targets_f)

# 4. Transform all input values
X_all_scaled_f = scaler_X_f.transform(
    df_processed_f[feature_cols_f]
).astype(np.float32)

# 5. Transform observed target values only
y_all_scaled_f = np.full(
    len(df_processed_f),
    np.nan,
    dtype=np.float32
)

valid_mask_f = (
    df_processed_f["target_original"]
    .notna()
    .to_numpy()
)

y_all_scaled_f[valid_mask_f] = (
    scaler_y_f.transform(
        df_processed_f.loc[
            valid_mask_f,
            ["target_original"]
        ]
    )
    .ravel()
    .astype(np.float32)
)

# 6. Save scalers
joblib.dump(
    scaler_X_f,
    os.path.join(experiment_dir_f, "scaler_X.joblib")
)

joblib.dump(
    scaler_y_f,
    os.path.join(experiment_dir_f, "scaler_y.joblib")
)

# 7. Verify
print("\nScaled X shape:", X_all_scaled_f.shape)
print("Scaled y shape:", y_all_scaled_f.shape)

print("\nMissing input values:",
      np.isnan(X_all_scaled_f).sum())

print("Missing target values:",
      np.isnan(y_all_scaled_f).sum())

print("\nTraining target minimum:",
      scaler_y_f.data_min_[0])

print("Training target maximum:",
      scaler_y_f.data_max_[0])

assert X_all_scaled_f.shape == (8016, 1)
assert y_all_scaled_f.shape == (8016,)
assert np.isnan(X_all_scaled_f).sum() == 0
assert np.isnan(y_all_scaled_f).sum() == 206

print("\nScalers saved successfully.")
print("FEATURE SCALING COMPLETED")


## 5. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 5
# GENERATE LSTM SEQUENCES (24 HOURS -> NEXT 1 HOUR)
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("EXPERIMENT F - LSTM SEQUENCE GENERATION")
print("=" * 70)

LOOKBACK_F = 24

# Prepare containers
X_train_list_f, y_train_list_f, time_train_list_f = [], [], []
X_val_list_f, y_val_list_f, time_val_list_f = [], [], []
X_test_list_f, y_test_list_f, time_test_list_f = [], [], []

skipped_missing_f = 0

timestamps_f = pd.to_datetime(
    df_processed_f["date_hour"]
).to_numpy()

# Generate sequences chronologically
for target_idx in range(LOOKBACK_F, len(df_processed_f)):

    input_start = target_idx - LOOKBACK_F

    X_window = X_all_scaled_f[input_start:target_idx]
    y_target = y_all_scaled_f[target_idx]

    # Skip missing actual targets
    if np.isnan(y_target):
        skipped_missing_f += 1
        continue

    # Verify hourly continuity
    window_times = pd.DatetimeIndex(
        timestamps_f[input_start:target_idx + 1]
    )

    if not (
        window_times.to_series().diff().iloc[1:]
        == pd.Timedelta(hours=1)
    ).all():
        continue

    target_time = pd.Timestamp(timestamps_f[target_idx])

    # Assign based on target timestamp
    if target_idx < train_end_f:

        X_train_list_f.append(X_window)
        y_train_list_f.append(y_target)
        time_train_list_f.append(target_time)

    elif target_idx < val_end_f:

        X_val_list_f.append(X_window)
        y_val_list_f.append(y_target)
        time_val_list_f.append(target_time)

    else:

        X_test_list_f.append(X_window)
        y_test_list_f.append(y_target)
        time_test_list_f.append(target_time)

# Convert lists to arrays
def convert_sequences_f(X_list, y_list):

    X = np.asarray(
        X_list, dtype=np.float32
    ).reshape(-1, LOOKBACK_F, 1)

    y = np.asarray(
        y_list, dtype=np.float32
    ).reshape(-1, 1)

    return X, y

X_train_f, y_train_f = convert_sequences_f(
    X_train_list_f, y_train_list_f
)

X_val_f, y_val_f = convert_sequences_f(
    X_val_list_f, y_val_list_f
)

X_test_f, y_test_f = convert_sequences_f(
    X_test_list_f, y_test_list_f
)

# Preserve timestamps
time_train_f = pd.DatetimeIndex(time_train_list_f)
time_val_f = pd.DatetimeIndex(time_val_list_f)
time_test_f = pd.DatetimeIndex(time_test_list_f)

# Display results
for name, X, y, times in [
    ("Training", X_train_f, y_train_f, time_train_f),
    ("Validation", X_val_f, y_val_f, time_val_f),
    ("Testing", X_test_f, y_test_f, time_test_f)
]:
    print(f"\n{name}")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("First target:", times.min())
    print("Last target:", times.max())

print("\nSkipped missing targets:", skipped_missing_f)

# Verify sequence dimensions
assert X_train_f.shape[1:] == (24, 1)
assert X_val_f.shape[1:] == (24, 1)
assert X_test_f.shape[1:] == (24, 1)

assert np.isfinite(X_train_f).all()
assert np.isfinite(y_train_f).all()
assert np.isfinite(X_val_f).all()
assert np.isfinite(y_val_f).all()
assert np.isfinite(X_test_f).all()
assert np.isfinite(y_test_f).all()

print("\nLSTM sequence generation completed successfully.")


## 6. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 6
# BUILD HISTORICAL-DEMAND-ONLY LSTM
# ============================================================

import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    Dense
)
from tensorflow.keras.optimizers import Adam

print("=" * 70)
print("EXPERIMENT F - LSTM MODEL ARCHITECTURE")
print("=" * 70)

# 1. Set random seed for reproducibility
tf.keras.utils.set_random_seed(42)

# 2. Define model
model_lstm_f = Sequential([
    Input(shape=(24, 1)),

    LSTM(64, return_sequences=True),
    Dropout(0.2),

    LSTM(32, return_sequences=True),
    Dropout(0.2),

    LSTM(16),
    Dropout(0.2),

    Dense(1)
])

# 3. Compile model
model_lstm_f.compile(
    optimizer=Adam(
        learning_rate=0.001,
        clipnorm=1.0
    ),
    loss="mse",
    metrics=["mae"]
)

# 4. Display model architecture
model_lstm_f.summary()

print("\nModel input shape:",
      model_lstm_f.input_shape)

print("Model output shape:",
      model_lstm_f.output_shape)

print("Total parameters:",
      model_lstm_f.count_params())

# 5. Verify input/output dimensions
assert model_lstm_f.input_shape == (None, 24, 1)
assert model_lstm_f.output_shape == (None, 1)

print("\nLSTM model architecture completed successfully.")


## 7. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 7
# TRAIN HISTORICAL-DEMAND-ONLY LSTM
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

print("=" * 70)
print("EXPERIMENT F - LSTM MODEL TRAINING")
print("=" * 70)

# 1. Verify training and validation data
print("\nTraining X:", X_train_f.shape)
print("Training y:", y_train_f.shape)
print("Validation X:", X_val_f.shape)
print("Validation y:", y_val_f.shape)

assert len(X_train_f) == len(y_train_f)
assert len(X_val_f) == len(y_val_f)

assert np.isfinite(X_train_f).all()
assert np.isfinite(y_train_f).all()
assert np.isfinite(X_val_f).all()
assert np.isfinite(y_val_f).all()

# 2. Define model checkpoint path
model_path_f = os.path.join(
    experiment_dir_f,
    "lstm_experiment_f_best.keras"
)

# 3. Define callbacks
early_stopping_f = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

model_checkpoint_f = ModelCheckpoint(
    filepath=model_path_f,
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

reduce_lr_f = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    mode="min",
    verbose=1
)

# 4. Train LSTM
history_f = model_lstm_f.fit(
    X_train_f,
    y_train_f,

    validation_data=(
        X_val_f,
        y_val_f
    ),

    epochs=200,
    batch_size=32,
    shuffle=False,

    callbacks=[
        early_stopping_f,
        model_checkpoint_f,
        reduce_lr_f
    ],

    verbose=1
)

# 5. Save training history
history_df_f = pd.DataFrame(
    history_f.history
)

history_path_f = os.path.join(
    experiment_dir_f,
    "training_history.csv"
)

history_df_f.to_csv(
    history_path_f,
    index=False
)

# 6. Display training summary
best_epoch_f = (
    np.argmin(history_f.history["val_loss"]) + 1
)

best_val_loss_f = np.min(
    history_f.history["val_loss"]
)

print("\n" + "=" * 70)
print("EXPERIMENT F - TRAINING SUMMARY")
print("=" * 70)

print("Total epochs trained:",
      len(history_f.history["loss"]))

print("Best epoch:", best_epoch_f)

print("Best validation MSE (scaled):",
      round(float(best_val_loss_f), 6))

print("\nBest model saved to:")
print(model_path_f)

print("\nTraining history saved to:")
print(history_path_f)

print("\nLSTM TRAINING COMPLETED")


## 8. ============================================================

In [ ]:

# ============================================================
# EXPERIMENT F - STEP 8
# LSTM MODEL EVALUATION
# HISTORICAL CHARGING DEMAND ONLY
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("=" * 70)
print("EXPERIMENT F - LSTM MODEL EVALUATION")
print("=" * 70)

# 1. Load best saved model
best_model_f = tf.keras.models.load_model(
    model_path_f
)

# 2. Evaluation function
def evaluate_lstm_f(model, X, y, timestamps, split_name):

    # Generate predictions (scaled)
    y_pred_scaled = model.predict(
        X,
        verbose=0
    )

    # Convert predictions to original kWh
    y_pred = scaler_y_f.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()

    y_actual = scaler_y_f.inverse_transform(
        y.reshape(-1, 1)
    ).ravel()

    # Calculate metrics
    rmse = np.sqrt(
        mean_squared_error(y_actual, y_pred)
    )

    mae = mean_absolute_error(
        y_actual, y_pred
    )

    r2 = r2_score(
        y_actual, y_pred
    )

    # Store prediction results
    prediction_df = pd.DataFrame({
        "date_hour": timestamps,
        "actual_kwh": y_actual,
        "predicted_kwh": y_pred
    })

    prediction_df["error_kwh"] = (
        prediction_df["predicted_kwh"]
        - prediction_df["actual_kwh"]
    )

    return {
        "Dataset": split_name,
        "Samples": len(y_actual),
        "RMSE (kWh)": rmse,
        "MAE (kWh)": mae,
        "R² Score": r2
    }, prediction_df


# 3. Evaluate all datasets
metrics_train_f, predictions_train_f = evaluate_lstm_f(
    best_model_f,
    X_train_f,
    y_train_f,
    time_train_f,
    "Training"
)

metrics_val_f, predictions_val_f = evaluate_lstm_f(
    best_model_f,
    X_val_f,
    y_val_f,
    time_val_f,
    "Validation"
)

metrics_test_f, predictions_test_f = evaluate_lstm_f(
    best_model_f,
    X_test_f,
    y_test_f,
    time_test_f,
    "Testing"
)

# 4. Create summary table
metrics_df_f = pd.DataFrame([
    metrics_train_f,
    metrics_val_f,
    metrics_test_f
])

print("\n" + "=" * 70)
print("EXPERIMENT F - PERFORMANCE SUMMARY")
print("=" * 70)

print(
    metrics_df_f.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# 5. Save evaluation results
metrics_df_f.to_csv(
    os.path.join(
        experiment_dir_f,
        "all_split_metrics.csv"
    ),
    index=False
)

predictions_train_f.to_csv(
    os.path.join(
        experiment_dir_f,
        "training_predictions.csv"
    ),
    index=False
)

predictions_val_f.to_csv(
    os.path.join(
        experiment_dir_f,
        "validation_predictions.csv"
    ),
    index=False
)

predictions_test_f.to_csv(
    os.path.join(
        experiment_dir_f,
        "test_predictions.csv"
    ),
    index=False
)

# 6. Display testing results
print("\n" + "=" * 70)
print("EXPERIMENT F - TESTING PERFORMANCE")
print("=" * 70)

print(
    f"RMSE: {metrics_test_f['RMSE (kWh)']:.4f} kWh"
)

print(
    f"MAE : {metrics_test_f['MAE (kWh)']:.4f} kWh"
)

print(
    f"R²  : {metrics_test_f['R² Score']:.4f}"
)

print("\nEvaluation results saved successfully.")
print("EXPERIMENT F - EVALUATION COMPLETED")
